**Imports And Config**

In [ ]:
import os
import sys
import matplotlib.pyplot as plt
from tqdm import tqdm

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

ROOT_DIR = os.path.abspath("../")
print('Root directory:',ROOT_DIR)
sys.path.append(ROOT_DIR)


from mal.utils import (
    load_image,
    extract_annotations,
)
from tdmms.tdmcoco import CocoConfig

from mrcnn import model as modellib

class malConfig(CocoConfig):
    # Set batch size to 1 since we'll be running inference on
    # one image at a time. Batch size = GPU_COUNT * IMAGES_PER_GPU
    GPU_COUNT = 1
    IMAGES_PER_GPU = 1
    DETECTION_MIN_CONFIDENCE = 0

DEFAULT_LOGS_DIR = os.path.join(ROOT_DIR, 'logs', 'area')

if not os.path.exists(DEFAULT_LOGS_DIR):
    os.makedirs(DEFAULT_LOGS_DIR)
    print(f"Folder '{DEFAULT_LOGS_DIR}' created.")

Root directory: c:\Users\gtgot\OneDrive\ML_Project\tdmms_DL


: 

**Extract Areas**

In [ ]:
area_list = []
count_list = []

config = malConfig()
model = modellib.MaskRCNN(
    mode="inference",
    config=config,
    model_dir=DEFAULT_LOGS_DIR
)

weights = 'graphene_mask_rcnn_tdm_0120.h5'

ROOT_DIR = os.path.abspath("../../")

MODEL_PATH = os.path.join(ROOT_DIR, 'weights', weights)
model.load_weights(MODEL_PATH, by_name=True)

for i in tqdm(os.listdir(os.path.join(ROOT_DIR, 'data', 'images', 'batch1'))):
    # print(f'Image: {i}')

    area = i.split('_')[-1].split('.')[0]        
    image = load_image(os.path.join(ROOT_DIR, 'data', 'images', 'batch1', i))

    results = model.detect([image])
    results = results[0]

    annotations = extract_annotations(
        results['rois'],
        results['masks'],
        results['class_ids'],
        ['','Mono', 'Few','Thick'],
        results['scores'],
        sieve_amount=10
    )
    
    if annotations:
        amount = len(annotations)
    else:
        amount = 0
    
    area_list.append(area)
    count_list.append(amount)

Instructions for updating:
Use fn_output_signature instead


  0%|          | 0/10 [00:00<?, ?it/s]c:\Users\gtgot\OneDrive\ML_Project\env_final\lib\site-packages\tensorflow\python\keras\engine\training.py:2325: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  warnings.warn('`Model.state_updates` will be removed in a future version. '
 10%|█         | 1/10 [00:08<01:12,  8.05s/it]


*** No instances to extract *** 



 20%|██        | 2/10 [00:14<00:57,  7.17s/it]


*** No instances to extract *** 



 40%|████      | 4/10 [00:28<00:42,  7.08s/it]

**Result**

In [ ]:
import numpy as np
np.linspace(0,10,10)

array([ 0.        ,  1.11111111,  2.22222222,  3.33333333,  4.44444444,
        5.55555556,  6.66666667,  7.77777778,  8.88888889, 10.        ])

: 

In [ ]:
area_list = [int(i) for i in area_list]
count_list = [int(i) for i in count_list]

area_list_sorted = sorted(area_list)
count_list_sorted = [x for _,x in sorted(zip(area_list,count_list))]

In [ ]:
threshold = 2000

plt.figure()
plt.scatter(area_list_sorted, count_list_sorted)
plt.xticks(np.linspace(area_list_sorted[0], area_list_sorted[-1], 5))
plt.xscale('log')
plt.axvline(threshold, color='red', label='2000 px^2')
plt.legend()
plt.ylabel('Annotations detected')
plt.xlabel('Area [px^2]')
plt.savefig(os.path.join(ROOT_DIR, 'images', 'area_analysis.eps'), format='eps')
plt.savefig(os.path.join(ROOT_DIR, 'images', 'area_analysis.png'))
plt.show()

print('Minimum area: {} px^2'.format(threshold))
print('sqrt({}) = {}'.format(threshold, np.sqrt(threshold)))